# 01 - Explore ERA5 Subset

This notebook demonstrates how to use the pipeline modules interactively.

Steps:
1. Load configuration
2. Download a GRIB file from Azure
3. Open it lazily with xarray
4. Inspect available variables and coordinates
5. Apply subset filters
6. Preview as pandas DataFrame

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import from src/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

In [ ]:
from src.config import load_config

config = load_config()
print("Years:", config.selection.years)
print("Variables:", config.selection.variables)
print("Time range:", config.selection.time_start, "to", config.selection.time_end)
print("Bounding box: lat", config.selection.lat_min, "to", config.selection.lat_max,
      "lon", config.selection.lon_min, "to", config.selection.lon_max)

## Download GRIB from Azure

This downloads the configured years to `data/raw/`. Skips if already cached.

In [ ]:
from src.storage import download_selected_blobs

local_paths = download_selected_blobs(config)
print("Downloaded files:", local_paths)

## Open dataset (lazy)

Nothing is loaded into memory here. xarray + dask keeps everything lazy.

In [ ]:
from src.loader import open_era5_dataset

ds = open_era5_dataset(local_paths[0], config)
ds

## Inspect available variables and coordinates

Use this to verify which variable names are in your GRIB file.
Update `config.yaml` if the names differ from the defaults.

In [ ]:
print("Variables:", list(ds.data_vars))
print("Coordinates:", list(ds.coords))
print("Dimensions:", dict(ds.dims))

## Apply subset filters

Filters by variables, time range, and bounding box. Still lazy.

In [ ]:
from src.subset import apply_all_filters

ds_sub = apply_all_filters(ds, config)
ds_sub

## Optional: Resample to daily

In [ ]:
from src.aggregate import resample_dataset

ds_daily = resample_dataset(ds_sub, config)
ds_daily

## Convert to pandas DataFrame

This is where data is actually loaded into memory.
Only do this after subsetting to a small slice.

In [ ]:
from src.export import to_dataframe

df = to_dataframe(ds_daily)
print("Shape:", df.shape)
df.head(10)

## Export to Zarr / Parquet

Uncomment and run to save the subset to disk.

In [ ]:
# from src.export import export_dataset
# results = export_dataset(ds_daily, config)
# print("Exported:", results)